In [1]:
from scipy.io import FortranFile 
import numpy as np 

In [106]:
def solve_block_lower(P,b):
    n = P.shape[0]
    bsize = np.zeros((n),dtype=int)
    bsize[:] = 1

    i = 0
    while i < n-1:
        if abs(P[i,i+1]) > 1.0e-12:
            bsize[i] = 2
            bsize[i+1] = 0
            i += 2
        else:
            i += 1
    
    # solve linear system 
    x = b * 0.
    i = 0
    while i < n:
        idx = np.arange(0,i)
        if bsize[i] == 1:
            s = np.sum(P[i,idx] * x[idx])
            if abs(P[i,i]) < 1.0e-12:
                x[i] = 0
                print(P[i,i],i,x[i])
            else:
                x[i] = (b[i] - s) / P[i,i]

            i += 1
            
        else:
            rhs1 = b[i] - np.sum(P[i,idx] * x[idx])
            rhs2 = b[i+1] - np.sum(P[i+1,idx] * x[idx])
            a11 = P[i,i]; a12 = P[i,i+1]
            a21 = P[i+1,i]; a22 = P[i+1,i+1]
            det = a11 * a22 - a12 * a21
            if abs(det) > 1.0e-12:
                x[i] = (a22 * rhs1 - a12 * rhs2) / det
                x[i+1] = (-a21 * rhs1 + a11 * rhs2) / det
            else:
                x[i] = 0. 
                x[i+1] = 0.
            i += 2 
    return x,bsize 
    

In [107]:
def read_file():
    fio = FortranFile("test.bin")
    P = fio.read_reals('f4')
    b = fio.read_reals('f4')
    x_obj = fio.read_reals('f4')
    fio.close()
    n = len(b)

    P = np.reshape(P,(n,n)).T 

    return P,b,x_obj

In [108]:
P,b,x_obj = read_file()

In [109]:
x,bsize = solve_block_lower(P,b)
print(x - x_obj)

0.0 33 0.0
[  -0.      -0.      -0.      -0.       0.      -0.       0.001    0.
    0.      -0.      -0.      -0.       0.015    0.      -0.001   -0.
   -0.       0.001    0.001   -0.       0.       0.       0.       0.001
   -0.      -0.       0.001   -0.       0.001   -0.      -0.      -0.
    0.001    9.838   -0.581   -1.606   -6.162   12.978    4.493   -6.108
    5.086  -31.23    -8.667   77.187  -78.425    8.857 -160.856   70.262
   15.766   37.232   94.054  -51.905  -50.944  106.823   26.85   -93.902
  -15.388   -7.694   20.99    88.501   50.807   87.819   42.728  -37.644
  106.059  -36.271  145.839  915.695  -87.449 -305.754]


In [116]:
x_obj - np.linalg.solve(P.T @ P,P.T @ b)

array([ -0.053,   0.011,  -0.083,   0.16 ,   0.061,  -0.019,   0.47 ,
         0.001,  -0.015,   0.008,   0.047,   0.185,  -8.91 ,  -0.027,
         0.393,   0.229,   0.189,  -0.372,  -0.638,   0.211,  -0.048,
        -0.089,   0.034,  -0.562,   0.232,   0.273,   0.071,  -0.233,
        -6.725,   0.705,  -1.949,   5.412,  -6.779, -10.191,  -1.971,
       -12.128,   7.633,   7.576,  -7.665,   5.288,   2.899,   3.774,
       -32.23 , -65.428,  37.813, -56.19 ,  79.697, -29.693, -35.977,
        13.125, -48.254,  11.599,   6.032, -10.697,  22.732,   7.22 ,
       -21.134,  -2.17 ,   1.283,  -5.072, -11.043,  -4.351,  -7.429,
         5.064,  -6.817,   3.889, -11.245, -79.367,   6.201,  22.596],
      dtype=float32)

In [88]:
np.set_printoptions(precision=3,suppress=True)
P[63:,63:]

array([[-13.304,   0.   ,   0.   ,   0.   ,   0.   ,   0.   ,   0.   ],
       [ -3.427,  -9.304,   0.   ,   0.   ,   0.   ,   0.   ,   0.   ],
       [ -2.405,  -2.807,  -9.872,   0.   ,   0.   ,   0.   ,   0.   ],
       [ -6.54 ,   4.867,  -5.963,  -9.021,   0.   ,   0.   ,   0.   ],
       [ -6.297,   4.013,   2.65 ,   9.995,  -2.351,   0.   ,   0.   ],
       [ -3.168,   7.764,   5.674,   0.859,  -1.532, -10.869,   0.   ],
       [ 11.806,   2.838,   6.75 ,   6.926,  -2.791,   7.143,  -9.934]],
      dtype=float32)

In [117]:
P @ x - b

array([ 0.   , -0.   , -0.   , -0.   ,  0.   ,  0.   ,  0.   ,  0.   ,
        0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   , -0.   , -0.   ,
        0.   ,  0.   , -0.   ,  0.   , -0.   , -0.   , -0.   , -0.   ,
        0.   , -0.   ,  0.   ,  0.   , -0.   ,  0.   ,  0.   ,  0.   ,
        0.   ,  0.   ,  0.   ,  0.   ,  0.   , -0.   , -0.   ,  0.   ,
        0.   , -0.   ,  0.   ,  0.   , -0.   , -0.   ,  0.   , -0.001,
       -0.   ,  0.   ,  0.   , -0.   , -0.   ,  0.   ,  0.   ,  0.   ,
       -0.   ,  0.   ,  0.   ,  0.001,  0.001,  0.   ,  0.001,  0.   ,
        0.   ,  0.   , -0.   , -0.001, -0.   ,  0.   ], dtype=float32)

In [94]:
x

array([  -0.152,    0.025,   -0.194,    0.314,    0.1  ,   -0.036,
          0.815,    0.015,    0.009,    0.062,    0.102,    0.34 ,
         -7.178,    0.009,    0.248,    0.113,    0.16 ,   -0.321,
         -0.641,    0.087,   -0.359,    0.146,   -0.444,   -1.384,
          0.606,    1.588,   -0.676,    0.622,   -5.746,    1.504,
         -0.969,    2.894,   -7.472,    0.   ,   -5.895,   -6.313,
          1.765,   19.086,   11.314,    8.305,  -13.494,    5.609,
        -46.343,    5.283,  -45.651,  -55.922,  -95.361,   41.277,
        -22.301,   64.509,   50.585,  -31.501,  -48.726,   86.498,
         55.88 ,  -78.965,  -41.397,   -6.521,   21.498,   91.827,
         22.754,   89.307,   18.682,  -33.195,  102.782,  -14.244,
        139.821,  915.695,  -60.989, -260.032], dtype=float32)

In [113]:
P[31:35,31:35]

array([[-19.05 ,   4.118,   0.   ,   0.   ],
       [ -5.542,   0.488,   0.   ,   0.   ],
       [-15.551,   1.877,   0.   ,   0.   ],
       [ 15.142,  -2.355,   1.666,  -0.075]], dtype=float32)